In [45]:
import os
import pickle
import requests
from bs4 import BeautifulSoup
from langchain_openai import ChatOpenAI
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [46]:
from dotenv import load_dotenv
import os

load_dotenv()

llm = ChatOpenAI(
    api_key="dummy",  # required by LangChain but not used by Sarvam
    base_url="https://api.sarvam.ai/v1",
    model="sarvam-30b",  # upgraded from legacy sarvam-m
    temperature=0.9,
    max_tokens=500,
    default_headers={
        "api-subscription-key": os.getenv("SARVAM_API_KEY")  # ← correct header
    }
)

In [50]:
from dotenv import load_dotenv, find_dotenv
import os

load_dotenv(find_dotenv(), override=True)

key = os.getenv("SARVAM_API_KEY")
print(f"Key starts with: {key[:15]}...")  # confirm new key loaded

Key starts with: sk_loexc9ye_OZX...


In [51]:
response = requests.post(
    "https://api.sarvam.ai/v1/chat/completions",
    headers={
        "api-subscription-key": key,
        "Content-Type": "application/json"
    },
    json={
        "model": "sarvam-30b",
        "messages": [{"role": "user", "content": "hello"}]
    }
)

print(response.status_code)  # should print 200

200


In [53]:
urls = [
    "https://www.moneycontrol.com/news/business/markets/wall-street-rises-as-tesla-soars-on-ai-optimism-11351111.html",
    "https://www.moneycontrol.com/news/business/tata-motors-launches-punch-icng-price-starts-at-rs-7-1-lakh-11098751.html"
]

headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"}

data = []
for url in urls:
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.text, "html.parser")
    text = soup.get_text(separator="\n", strip=True)
    data.append(Document(page_content=text, metadata={"source": url}))

print(f"Loaded {len(data)} documents")

Loaded 2 documents


In [54]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
docs = splitter.split_documents(data)
print(f"Total chunks: {len(docs)}")

Total chunks: 29


In [55]:
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorIndex = FAISS.from_documents(docs, embeddings)

# Save to disk
with open("vector_index.pkl", "wb") as f:
    pickle.dump(vectorIndex, f)

print("vectorIndex is ready!")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3092.15it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


vectorIndex is ready!


In [56]:
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorIndex = FAISS.from_documents(docs, embeddings)

# Save to disk
with open("vector_index.pkl", "wb") as f:
    pickle.dump(vectorIndex, f)

print("vectorIndex is ready!")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2687.80it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


vectorIndex is ready!


In [57]:
retriever = vectorIndex.as_retriever()

prompt = ChatPromptTemplate.from_template("""
Answer the question based only on the following context.
Always mention the source URL at the end.

Context: {context}

Question: {question}
""")

chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

query = "what is the price of Tiago iCNG?"
result = chain.invoke(query)
print(result)

Based on the provided context, the price of the Tiago iCNG is not mentioned.

However, the context does specify the price of the Tata Motors **Punch iCNG**, which is stated to be **Rs 7.1 lakh**.

Source: https://www.moneycontrol.com/news/business/tata-motors-launches-punch-icng-price-starts-at-rs-7-1-lakh-11098751.html
